# Superdense coding
**Quantum Computing · Week 5 (Quantum protocols and algorithms) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w5d2.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Superdense coding** sends two classical bits by sending a single qubit, as long as Alice and Bob
shared a Bell pair beforehand. It's teleportation run the other way round.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, partial_trace
from qiskit.primitives import StatevectorSampler

def encode(message):
    """The shared Bell pair, then Alice's gate on her qubit q0. message = two bits, as Qiskit prints them."""
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)                    # the Bell pair: q0 for Alice, q1 for Bob
    qc.barrier()
    if message[0] == "1":
        qc.x(0)                    # the left bit decides X...
    if message[1] == "1":
        qc.z(0)                    # ...and the right bit decides Z
    qc.barrier()
    return qc

def superdense(message):
    qc = encode(message)
    qc.cx(0, 1)                    # Bob, after receiving q0: the Bell circuit backwards
    qc.h(0)
    qc.measure_all()
    return qc

superdense("11").draw("mpl", style="clifford")

## Send all four messages

In [ ]:
sampler = StatevectorSampler(seed=1)
for message in ["00", "01", "10", "11"]:
    counts = sampler.run([superdense(message)], shots=1000).result()[0].data.meas.get_counts()
    print(f"Alice sends {message} -> Bob reads {counts}")

## Four messages, four Bell states
After Alice's gate the pair is one of the four Bell states. They are **orthogonal**: every pair
has overlap 0, which is exactly why Bob can tell them apart with certainty.

In [ ]:
states = {m: Statevector(encode(m)) for m in ["00", "01", "10", "11"]}
for m, sv in states.items():
    print(m, "->", np.round(sv.data.real, 3) + 0.0, "   (amplitudes of 00, 01, 10, 11)")

overlaps = np.array([[abs(states[a].inner(states[b])) ** 2 for b in states] for a in states])
print("\n|<a|b>|^2 for every pair of messages:")
print(np.round(overlaps, 3) + 0.0)

## Your turn: what does Eve see?
Eve catches the qubit on its way to Bob. Use `partial_trace` to compute the state of q0 alone
(trace out Bob's q1) for each of the four messages. Can she tell them apart?

In [ ]:
for message, sv in states.items():
    eve_view = None   # TODO: partial_trace(sv, [1]) keeps q0 and averages over q1
    print(message, eve_view)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for message, sv in states.items():
    eve_view = partial_trace(sv, [1])
    print(message, "Eve's qubit:", (np.round(eve_view.data.real, 3) + 0.0).tolist())
print("The same matrix I/2 for every message: Eve sees a fair coin and learns nothing.\n")

# Bonus: what if Eve measures the qubit? Then Bob's second bit becomes a coin flip.
from qiskit_aer import AerSimulator
from qiskit import ClassicalRegister

spied = encode("01")
eve = ClassicalRegister(1, "eve")
spied.add_register(eve)
spied.measure(0, eve[0])            # Eve measures q0 in the middle of the trip
spied.cx(0, 1)
spied.h(0)
spied.measure_all()
sim = AerSimulator(seed_simulator=3)
counts = sim.run(transpile(spied, sim), shots=1000).result().get_counts()
bob = {}
for key, n in counts.items():
    bob_bits = key.split()[0]       # registers print right to left: "meas eve"
    bob[bob_bits] = bob.get(bob_bits, 0) + n
print("Alice sent 01 with Eve listening; Bob reads:", bob)